# 1. Imports and setup

In [9]:
import os
import sys
import json
import hashlib
import time
from datetime import datetime, timezone
from pathlib import Path
from datetime import datetime

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from tqdm import tqdm

sys.path.append('..')
from src.prompter import Prompter



# Importacion para paralelizar? 
import concurrent.futures

# Importaciones para similitudes entre keywords
from difflib import SequenceMatcher
from collections import defaultdict

import re


# 2. Configuración

In [ ]:
# ── LLM config ──────────────────────────────────────────────────────────────
MODEL_TYPE   = "qwen3:32b"
#CONFIG_PATH  = "static/config/config.yaml"
CONFIG_PATH  = "../config.yaml"

# ── Query parameters ────────────────────────────────────────────────────────
ROOT_TOPIC   = "living lab"   # Starting topic
N_KEYWORDS   = 10         # Keywords requested per query
N_KEYWORDS_2 = 50         # Keywords requested per query for levels > 1
MAX_LEVELS   = 5          # How many levels deep to expand (0 = root only)

SIMILARITY_THRESHOLD = 0.6  # Threshold for considering two keywords as similar
# ── Cache config ────────────────────────────────────────────────────────────
CACHE_DIR    = Path("similarity_query_cache")
CACHE_DIR.mkdir(exist_ok=True)

# ── Initialise prompter ─────────────────────────────────────────────────────
prompter = Prompter(config_path=CONFIG_PATH, model_type=MODEL_TYPE, temperature=0.3)

src.prompter - INFO - Setting temperature to: 0.3
src.prompter - INFO - Using yiyuan host: https://yiyuan.tsc.uc3m.es
src.prompter - INFO - Using yiyuan API with host: https://yiyuan.tsc.uc3m.es


Loaded config file ../config.yaml and section logger.
Logs will be saved in data/logs
Loaded config file ../config.yaml and section llm.


# 3. Funciones de cache

In [ ]:
def _cache_key(topic: str, n_keywords: int) -> str:
    """Stable filename key for a (topic, n_keywords) pair."""
    raw = f"{topic.strip().lower()}_{n_keywords}"
    return hashlib.md5(raw.encode()).hexdigest()[:12] + f"__{topic.strip().lower().replace(' ', '_')}"


def cache_path(topic: str, n_keywords: int) -> Path:
    return CACHE_DIR / f"{_cache_key(topic, n_keywords)}.json"


def load_cache(topic: str, n_keywords: int) -> dict | None:
    """Return cached result dict or None if not found."""
    p = cache_path(topic, n_keywords)
    if p.exists():
        with open(p) as f:
            return json.load(f)
    return None


def save_cache(topic: str, n_keywords: int, raw_response: str, parsed: dict) -> None:
    record = {
        "topic": topic,
        "n_keywords": n_keywords,
        "timestamp": datetime.now(timezone.utc).isoformat(),  # fixed deprecation
        "raw_response": raw_response,
        "parsed": parsed,
    }
    with open(cache_path(topic, n_keywords), "w") as f:
        json.dump(record, f, indent=2)


def invalidate_cache(topic: str, n_keywords: int) -> None:
    """Delete a cached entry to force a fresh LLM call."""
    p = cache_path(topic, n_keywords)
    if p.exists():
        p.unlink()
        print(f"Cache cleared for '{topic}'.")
    else:
        print(f"No cache entry found for '{topic}'.")


def list_cache() -> pd.DataFrame:
    """Show every cached query as a DataFrame for easy review."""
    records = []
    for p in sorted(CACHE_DIR.glob("*.json")):
        with open(p) as f:
            rec = json.load(f)
        records.append({
            "file": p.name,
            "topic": rec["topic"],
            "n_keywords": rec["n_keywords"],
            "timestamp": rec["timestamp"],
            "n_parsed_keys": len(rec["parsed"]),
        })
    return pd.DataFrame(records) if records else pd.DataFrame(columns=["file","topic","n_keywords","timestamp","n_parsed_keys"])


print("Cache helpers loaded. Cache directory:", CACHE_DIR.resolve())

# 4. Funciones

In [8]:
# Función de similitud entre dos keywords
def is_similar(kwd1, kwd2, threshold = SIMILARITY_THRESHOLD):
    return SequenceMatcher(None, kwd1.lower(), kwd2.lower()).ratio() >= threshold

print(is_similar("living lab", "urban living-labs"))
print(is_similar("living lab", "integrated living-labs"))

True
False


In [ ]:
DOMAIN_DEFINITIONS = {
    "living lab": {
        "full": (
            "A Living Lab is a real-world open innovation ecosystem where citizens, researchers, "
            "companies, and governments co-create and test solutions in everyday life contexts."
        ),
        "dimensions": [
            "real-world testing and experimentation contexts",
            "open innovation processes and methodologies",
            "multi-actor collaboration (citizens, researchers, companies, governments)",
            "co-creation and participatory design methodologies",
            "everyday life application and urban/social contexts",
        ]
    }
}

In [ ]:
def build_prompt(topic: str, n_keywords: int, root_topic: str = "") -> str:
    is_root = not root_topic or topic == root_topic
    domain_def = DOMAIN_DEFINITIONS.get(topic if is_root else root_topic, None)

    if domain_def:
        if is_root:
            definition_block = f"\nTOPIC DEFINITION: {domain_def['full']}\n"
        else:
            dims = "\n".join(f"  - {d}" for d in domain_def["dimensions"])
            definition_block = f"""
                SCORING CRITERIA: A keyword scores 8-10 only if it directly relates to at least 
                one of these dimensions:
                {dims}
                Keywords unrelated to these dimensions should score 1-3 at most.
                """
    else:
        definition_block = ""

    return f"""You are a JSON-only output machine. You never write explanations, greetings, or markdown.
                TASK: Given the topic "{topic}", return exactly {n_keywords} related keywords with specificity scores.
                {definition_block}
                RULES (violations will break the system):
                1. Output MUST start with {{ and end with }} — nothing before, nothing after
                2. The topic itself must appear first with score 0
                3. No keyword may contain the topic word "{topic}"
                4. All keywords must be unique
                5. Scores are integers 1-10 only (topic gets 0)
                6. No markdown, no ```json, no explanations, no trailing text
                SCORING:
                8-10 → direct synonyms or highly specific sub-concepts
                4-7  → broadly related fields or associated concepts  
                1-3  → loose umbrella terms
                TOPIC: {topic}
                OUTPUT (raw JSON only):
                {{
                "{topic}": 0,
                "keyword1": score,
                "keyword2": score
                }}"""

In [ ]:
def extract_dictionary(response: str) -> dict:
    """Strip optional markdown fences and parse JSON."""
    #clean = response.strip()
    clean = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()

    if clean.startswith("```json"):
        clean = clean[7:]
    elif clean.startswith("```"):
        clean = clean[3:]
    if clean.endswith("```"):
        clean = clean[:-3]
    return json.loads(clean.strip())

def validate_keywords(parsed: dict, topic: str) -> None:
    """Raise ValueError if the parsed dict violates the expected contract."""

    if topic not in parsed:
        raise ValueError(f"Topic '{topic}' missing. Got: {list(parsed.keys())}")
    if parsed[topic] != 0:
        raise ValueError(f"Topic '{topic}' has score {parsed[topic]}, expected 0")
    if len(parsed) < 2:
        raise ValueError(f"Only {len(parsed)} entries returned, expected at least 2")

    topic_lower = topic.lower()
    for kwd, score in parsed.items():
        if kwd == topic:
            continue
        if not isinstance(score, int):
            raise ValueError(f"Score for '{kwd}' is {type(score).__name__}, expected int")
        if not (1 <= score <= 10):
            raise ValueError(f"Score {score} for '{kwd}' is out of range [1, 10]")
        # Only block if the FULL topic phrase appears inside the keyword
        if topic_lower in kwd.lower():
            raise ValueError(f"Keyword '{kwd}' contains the full topic phrase '{topic}'")



def get_keywords(topic: str, n_keywords: int, prompter,
                 root_topic: str = "",
                 use_cache: bool = True, max_retries: int = 3) -> dict:
    
    if use_cache:
        cached = load_cache(topic, n_keywords)
        if cached is not None:
            return cached["parsed"]

    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            prompt = build_prompt(topic, n_keywords, root_topic=root_topic)
            
            print(prompt)
            
            raw    = prompter.prompt(question=prompt, system_prompt_template_path=None)[0]
            
            print("// Raw LLM response:")
            
            print(raw)
            
            parsed = extract_dictionary(raw)
            
            # Validate the result makes sense before accepting it
            validate_keywords(parsed, topic)
            
            save_cache(topic, n_keywords, raw, parsed)
            return parsed

        except Exception as e:
            last_error = e
            print(f"  Attempt {attempt}/{max_retries} failed for '{topic}': {e}")
            time.sleep(1 * attempt)  # simple backoff

    # Save the last bad response for inspection
    bad_path = CACHE_DIR / f"FAILED__{topic.replace(' ', '_')}.txt"
    bad_path.write_text(raw)
    raise RuntimeError(
        f"All {max_retries} attempts failed for '{topic}'. "
        f"Last error: {last_error}. Raw response saved to {bad_path}"
    )

def df_from_dict(dictionary: dict, level: int) -> pd.DataFrame:
    """
    Convert a keyword dict into a DataFrame row-set.

    - The topic entry (score == 0) is included only at level 0
      (it is already present from a parent call at deeper levels).
    - All other entries become children of the root keyword.
    """
    root_kwd = next(k for k, v in dictionary.items() if v == 0)

    rows = []
    for kwrd, score in dictionary.items():
        if score == 0:
            if level == 0:          # include root node only at the first level
                rows.append({"kwrd": kwrd, "lvl": 0, "parent": None, "score": 0})
        else:
            rows.append({"kwrd": kwrd, "lvl": level + 1, "parent": root_kwd, "score": score})

    return pd.DataFrame(rows)


print("Core functions defined.")

In [ ]:
def expand_keywords_parallel(level_keywords: list, lvl: int, root_topic: str, prompter):
    results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        future_to_kwd = {
            executor.submit(get_keywords_with_ranks, kwd, N_KEYWORDS_2, prompter, root_topic): kwd
            for kwd in level_keywords
        }
        for future in concurrent.futures.as_completed(future_to_kwd):
            kwd = future_to_kwd[future]
            try:
                # 'data' contiene {keyword: {"score": s, "position": p}}
                data = future.result()
                results.append((kwd, data))
            except Exception as exc:
                print(f"'{kwd}' generated an exception: {exc}")
    return results

In [ ]:
root_dict = get_keywords(topic=ROOT_TOPIC, n_keywords=N_KEYWORDS, prompter=prompter)
df = df_from_dict(root_dict, level=0)
df_rows = []
edges = [] 

current_level_nodes = [kw for kw in root_data.keys() if kw != ROOT_TOPIC]

for lvl in range(1, 3):
    level_keywords = df[df["lvl"] == lvl]["kwrd"].tolist()
    for kwd in tqdm(level_keywords, desc=f"Level {lvl} keywords"):
        #kwd_dict = get_keywords(kwd, N_KEYWORDS_2, prompter)
        #kwd_dict = get_keywords(topic=kwd, n_keywords=N_KEYWORDS_2, prompter=prompter, root_topic=ROOT_TOPIC)
        kwd_dict = get_keywords(topic=kwd, n_keywords=3, prompter=prompter, root_topic=ROOT_TOPIC)

        df_new   = df_from_dict(kwd_dict, level=lvl)
        df       = pd.concat([df, df_new], ignore_index=True)

print(f"\nDone! Tree has {len(df)} rows across {df['lvl'].max() + 1} levels.")
df